# Impacts of Timestep reduction

### Classification without RUS and SMOTE - (SVM, and GRU)

In [1]:
import pickle
import numpy as np
import os

SPLIT_DIR = "./final_split_data_HybridNorm_timesteps"
os.makedirs(SPLIT_DIR, exist_ok=True)

def load_split(path):
    with open(path, "rb") as f:
        data = pickle.load(f)
    return data["X"].astype(np.float32), data["y"]

# ── Load hybrid only ──────────────────────────────────────────
X_train_full, y_train = load_split("./final_split_data_HybridNorm/train_set.pkl")
X_val_full,   y_val   = load_split("./final_split_data_HybridNorm/val_set.pkl")
X_test_full,  y_test  = load_split("./final_split_data_HybridNorm/test_set.pkl")

# ── Three timestep variants ───────────────────────────────────
datasets = {
    "hybrid_288": {
        "X_train": X_train_full,              "y_train": y_train,
        "X_val":   X_val_full,                "y_val":   y_val,
        "X_test":  X_test_full,               "y_test":  y_test,
    },
    "hybrid_144": {
        "X_train": X_train_full[:, -144:, :], "y_train": y_train,
        "X_val":   X_val_full[:, -144:, :],   "y_val":   y_val,
        "X_test":  X_test_full[:, -144:, :],  "y_test":  y_test,
    },
    "hybrid_72": {
        "X_train": X_train_full[:, -72:, :],  "y_train": y_train,
        "X_val":   X_val_full[:, -72:, :],    "y_val":   y_val,
        "X_test":  X_test_full[:, -72:, :],   "y_test":  y_test,
    },
}

# ── Save ──────────────────────────────────────────────────────
for name, d in datasets.items():
    folder = os.path.join(SPLIT_DIR, name)
    os.makedirs(folder, exist_ok=True)

    for split in ["train", "val", "test"]:
        filepath = os.path.join(folder, f"{split}_set.pkl")
        with open(filepath, "wb") as f:
            pickle.dump({"X": d[f"X_{split}"], "y": d[f"y_{split}"]}, f)

    print(f"  [{name}]  train={d['X_train'].shape}  val={d['X_val'].shape}  test={d['X_test'].shape}  → {folder}/")

# ── Summary ───────────────────────────────────────────────────
print(f"\n{'Dataset':<15} {'Train shape':<25} {'Val shape':<25} {'Test shape':<25} {'Class 0':>8} {'Class 1':>8} {'Ratio':>8}")
print("─" * 120)
for name, d in datasets.items():
    counts = np.bincount(d["y_train"].astype(int))
    ratio  = counts[0] / counts[1]
    print(f"{name:<15} {str(d['X_train'].shape):<25} {str(d['X_val'].shape):<25} {str(d['X_test'].shape):<25} "
          f"{counts[0]:>8} {counts[1]:>8} {ratio:>7.1f}x")

print(f"\n⚠️  Class imbalance (~{ratio:.0f}:1) — use class_weight={{0:1, 1:{ratio:.0f}}} in GRU.")
print(f"\n✅  All saved to {os.path.abspath(SPLIT_DIR)}/")

  [hybrid_288]  train=(12455, 288, 10)  val=(1780, 288, 10)  test=(3559, 288, 10)  → ./final_split_data_HybridNorm_timesteps/hybrid_288/
  [hybrid_144]  train=(12455, 144, 10)  val=(1780, 144, 10)  test=(3559, 144, 10)  → ./final_split_data_HybridNorm_timesteps/hybrid_144/
  [hybrid_72]  train=(12455, 72, 10)  val=(1780, 72, 10)  test=(3559, 72, 10)  → ./final_split_data_HybridNorm_timesteps/hybrid_72/

Dataset         Train shape               Val shape                 Test shape                 Class 0  Class 1    Ratio
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
hybrid_288      (12455, 288, 10)          (1780, 288, 10)           (3559, 288, 10)              12337      118   104.6x
hybrid_144      (12455, 144, 10)          (1780, 144, 10)           (3559, 144, 10)              12337      118   104.6x
hybrid_72       (12455, 72, 10)           (1780, 72, 10)            (3559, 72, 10)               12337      1

### Vectorization for SVM

In [2]:
# ══════════════════════════════════════════════════════════════
# CATCH22 EXTRACTION — hybrid timestep variants
# ══════════════════════════════════════════════════════════════
import os
import numpy as np
from joblib import Parallel, delayed
from sktime.transformations.panel.catch22 import Catch22

SAVE_DIR = "./catch22_features_timesteps"
os.makedirs(SAVE_DIR, exist_ok=True)


def _catch22_chunk(X_chunk: np.ndarray) -> np.ndarray:
    transformer = Catch22(catch24=False)
    return transformer.fit_transform(X_chunk).to_numpy()


def extract_catch22(X: np.ndarray, label: str = "", n_jobs: int = -1, chunk_size: int = 500) -> np.ndarray:
    n_samples, n_timepoints, n_channels = X.shape

    if not np.isfinite(X).all():
        n_bad = (~np.isfinite(X)).sum()
        print(f"    ⚠️  {n_bad} non-finite values in {label} — replacing with per-channel median")
        X = X.copy()
        for c in range(n_channels):
            col = X[:, :, c]
            col[~np.isfinite(col)] = np.nanmedian(col)
            X[:, :, c] = col

    X_sktime = X.transpose(0, 2, 1).astype(np.float64)
    chunks   = [X_sktime[i:i + chunk_size] for i in range(0, n_samples, chunk_size)]
    print(f"    {label} — {n_samples} samples in {len(chunks)} chunks …", flush=True)

    results = Parallel(n_jobs=n_jobs)(
        delayed(_catch22_chunk)(chunk) for chunk in chunks
    )

    X_feat = np.vstack(results)

    expected_cols = 22 * n_channels
    assert X_feat.shape == (n_samples, expected_cols), (
        f"Unexpected catch22 output shape: {X_feat.shape}  "
        f"(expected ({n_samples}, {expected_cols}))"
    )
    return X_feat


# ── Sanity check ───────────────────────────────────────────────
print("Pre-extraction value range check:")
print(f"{'Dataset':<15} {'Finite?':<10} {'Min':>15} {'Max':>15} {'NaN count':>12}")
print("─" * 70)
for name, d in datasets.items():
    X = d["X_train"]
    print(f"{name:<15} {str(np.isfinite(X).all()):<10} {X.min():>15.4f} {X.max():>15.4f} {(~np.isfinite(X)).sum():>12}")

# ── Extraction + save loop ─────────────────────────────────────
print("\nExtracting & saving catch22 features …")

datasets_svm = {}

for name, d in datasets.items():
    print(f"\n[{name}] extracting train …", flush=True)
    X_tr = extract_catch22(d["X_train"], label=f"{name}/train", n_jobs=-1)

    print(f"[{name}] extracting val …", flush=True)
    X_va = extract_catch22(d["X_val"],   label=f"{name}/val",   n_jobs=-1)

    print(f"[{name}] extracting test …", flush=True)
    X_te = extract_catch22(d["X_test"],  label=f"{name}/test",  n_jobs=-1)

    np.savez(os.path.join(SAVE_DIR, f"{name}_train.npz"), X=X_tr, y=d["y_train"])
    np.savez(os.path.join(SAVE_DIR, f"{name}_val.npz"),   X=X_va, y=d["y_val"])
    np.savez(os.path.join(SAVE_DIR, f"{name}_test.npz"),  X=X_te, y=d["y_test"])

    datasets_svm[name] = {
        "X_train": X_tr, "y_train": d["y_train"],
        "X_val":   X_va, "y_val":   d["y_val"],
        "X_test":  X_te, "y_test":  d["y_test"],
    }

    print(f"[{name}] ✓  train {X_tr.shape}  val {X_va.shape}  test {X_te.shape}  → {SAVE_DIR}/")

print(f"\n✅  Catch22 extraction complete.")
print(f"    Folder : {os.path.abspath(SAVE_DIR)}/")
print(f"    Shape  : 22 features × 10 channels = 220 cols per sample")

Pre-extraction value range check:
Dataset         Finite?                Min             Max    NaN count
──────────────────────────────────────────────────────────────────────
hybrid_288      True               -6.5219        248.1381            0
hybrid_144      True               -6.5219        233.3031            0
hybrid_72       True               -6.5219         93.5328            0

Extracting & saving catch22 features …

[hybrid_288] extracting train …
    hybrid_288/train — 12455 samples in 63 chunks …
[hybrid_288] extracting val …
    hybrid_288/val — 1780 samples in 9 chunks …
[hybrid_288] extracting test …
    hybrid_288/test — 3559 samples in 18 chunks …
[hybrid_288] ✓  train (12455, 220)  val (1780, 220)  test (3559, 220)  → ./catch22_features_timesteps/

[hybrid_144] extracting train …
    hybrid_144/train — 12455 samples in 63 chunks …
[hybrid_144] extracting val …
    hybrid_144/val — 1780 samples in 9 chunks …
[hybrid_144] extracting test …
    hybrid_144/test — 3559

### Missing value imputation from Catch22 - Loading the data

In [3]:
from sklearn.impute import SimpleImputer
import os
import numpy as np

SAVE_DIR = "./catch22_features_timesteps"
datasets_svm = {}

for fname in sorted(os.listdir(SAVE_DIR)):
    if not fname.endswith("_train.npz"):
        continue
    name = fname.replace("_train.npz", "")

    paths = {
        "train": os.path.join(SAVE_DIR, f"{name}_train.npz"),
        "val":   os.path.join(SAVE_DIR, f"{name}_val.npz"),
        "test":  os.path.join(SAVE_DIR, f"{name}_test.npz"),
    }
    if not all(os.path.exists(p) for p in paths.values()):
        print(f"  ⚠️  [{name}] missing file — skipping")
        continue

    tr = np.load(paths["train"])
    va = np.load(paths["val"])
    te = np.load(paths["test"])

    X_tr, y_tr = tr["X"], tr["y"]
    X_va, y_va = va["X"], va["y"]
    X_te, y_te = te["X"], te["y"]

    n_bad_tr = (~np.isfinite(X_tr)).sum()
    n_bad_va = (~np.isfinite(X_va)).sum()
    n_bad_te = (~np.isfinite(X_te)).sum()

    if n_bad_tr > 0 or n_bad_va > 0 or n_bad_te > 0:
        print(f"  ⚠️  [{name}] non-finite values — "
              f"train: {n_bad_tr}  val: {n_bad_va}  test: {n_bad_te}  → imputing")

        X_tr = np.where(np.isfinite(X_tr), X_tr, np.nan)
        X_va = np.where(np.isfinite(X_va), X_va, np.nan)
        X_te = np.where(np.isfinite(X_te), X_te, np.nan)

        imp  = SimpleImputer(strategy="median")
        X_tr = imp.fit_transform(X_tr)
        X_va = imp.transform(X_va)
        X_te = imp.transform(X_te)

        np.savez(paths["train"], X=X_tr, y=y_tr)
        np.savez(paths["val"],   X=X_va, y=y_va)
        np.savez(paths["test"],  X=X_te, y=y_te)
        print(f"  ✓  [{name}] cleaned and saved")
    else:
        print(f"  ✓  [{name}] no NaN or Inf — files unchanged")

    datasets_svm[name] = {
        "X_train": X_tr, "y_train": y_tr,
        "X_val":   X_va, "y_val":   y_va,
        "X_test":  X_te, "y_test":  y_te,
    }

print(f"\n✅  datasets_svm loaded: {list(datasets_svm.keys())}")
for name, d in datasets_svm.items():
    print(f"    {name:<15}  train={d['X_train'].shape}  val={d['X_val'].shape}  test={d['X_test'].shape}")

  ⚠️  [hybrid_144] non-finite values — train: 976  val: 1020  test: 1012  → imputing
  ✓  [hybrid_144] cleaned and saved
  ⚠️  [hybrid_288] non-finite values — train: 148  val: 464  test: 200  → imputing
  ✓  [hybrid_288] cleaned and saved
  ⚠️  [hybrid_72] non-finite values — train: 1828  val: 1296  test: 1649  → imputing
  ✓  [hybrid_72] cleaned and saved

✅  datasets_svm loaded: ['hybrid_144', 'hybrid_288', 'hybrid_72']
    hybrid_144       train=(12455, 220)  val=(1780, 220)  test=(3559, 220)
    hybrid_288       train=(12455, 220)  val=(1780, 220)  test=(3559, 220)
    hybrid_72        train=(12455, 220)  val=(1780, 220)  test=(3559, 220)


### Classification

In [2]:
# ══════════════════════════════════════════════════════════════
# HELPER FUNCTIONS
# ══════════════════════════════════════════════════════════════
import numpy as np
from sklearn.metrics import confusion_matrix
from sklearn.svm import SVC
import time
import os

RESULTS_DIR = "./results"
os.makedirs(RESULTS_DIR, exist_ok=True)


def compute_metrics(y_true, y_pred):
    cm             = confusion_matrix(y_true, y_pred)
    TN, FP, FN, TP = cm.ravel()

    recall    = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    accuracy  = (TP + TN) / (TP + TN + FP + FN)

    tss  = recall - FP / (FP + TN) if (FP + TN) > 0 else 0.0

    expected = ((TP + FN) * (TP + FP) + (TN + FP) * (TN + FN)) / (TP + TN + FP + FN) ** 2
    hss1     = (accuracy - expected) / (1 - expected) if (1 - expected) > 0 else 0.0

    denom = ((TP + FN) * (FN + TN) + (TP + FP) * (FP + TN))
    hss2  = 2 * (TP * TN - FP * FN) / denom if denom > 0 else 0.0

    hits_random = (TP + FP) * (TP + FN) / (TP + TN + FP + FN)
    gss = (TP - hits_random) / (TP + FP + FN - hits_random) if (TP + FP + FN - hits_random) > 0 else 0.0

    return {
        'TP': int(TP), 'TN': int(TN), 'FP': int(FP), 'FN': int(FN),
        'tss': tss, 'hss1': hss1, 'hss2': hss2, 'gss': gss,
        'recall': recall, 'f1': f1, 'accuracy': accuracy
    }


def train_and_evaluate(model, X_train, y_train, X_test, y_test):
    t0         = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - t0

    t0         = time.time()
    y_pred     = model.predict(X_test)
    infer_time = time.time() - t0

    metrics = compute_metrics(y_test, y_pred)
    metrics['train_time'] = train_time
    metrics['infer_time'] = infer_time
    return metrics


def save_results(metrics_list, filepath):
    with open(filepath, 'w') as f:
        for m in metrics_list:
            line = (f"{m['TP']},{m['TN']},{m['FP']},{m['FN']},"
                    f"{m['tss']:.6f},{m['hss1']:.6f},{m['hss2']:.6f},{m['gss']:.6f},"
                    f"{m['recall']:.6f},{m['f1']:.6f},{m['accuracy']:.6f},"
                    f"{m['train_time']:.4f},{m['infer_time']:.4f}")
            f.write(line + "\n")


def print_results(metrics_list, title):
    keys = ['tss', 'hss1', 'hss2', 'gss', 'recall', 'f1', 'accuracy', 'train_time', 'infer_time']
    print(f"\n{'─'*55}")
    print(f"  {title}")
    print(f"{'─'*55}")
    for i, m in enumerate(metrics_list):
        print(f"  Run {i+1}: TP={m['TP']}  TN={m['TN']}  FP={m['FP']}  FN={m['FN']}")
        print(f"         TSS={m['tss']:.4f}  HSS1={m['hss1']:.4f}  HSS2={m['hss2']:.4f}  GSS={m['gss']:.4f}")
        print(f"         Recall={m['recall']:.4f}  F1={m['f1']:.4f}  Acc={m['accuracy']:.4f}")
        print(f"         Train={m['train_time']:.2f}s  Infer={m['infer_time']:.4f}s")
        print()
    print(f"  ── Average of {len(metrics_list)} runs ──")
    for k in keys:
        avg = np.mean([m[k] for m in metrics_list])
        print(f"  {k:<12} : {avg:.4f}")
    print(f"{'─'*55}")

### SVM

In [10]:
from sklearn.svm import LinearSVC

best_model_class = LinearSVC
best_params      = {"C": 1, "class_weight": "balanced", "max_iter": 3000}

DATASET_NAMES = {
    "hybrid_144" : "hybrid_144",
    "hybrid_72"  : "hybrid_72",
}

print(f"{'═'*60}")
print(f"  Classifier : SVM")
print(f"  Model      : LinearSVC  C=1  cw=balanced")
print(f"{'═'*60}")

for norm_key, norm_label in DATASET_NAMES.items():
    if norm_key not in datasets_svm:
        print(f"  [{norm_key}] not found — skipping")
        continue

    d = datasets_svm[norm_key]
    print(f"\n  ── Dataset: {norm_label} ──")

    metrics_list = []
    for run in range(2):
        model   = best_model_class(**best_params)
        metrics = train_and_evaluate(
            model,
            d["X_train"], d["y_train"],
            d["X_test"],  d["y_test"]
        )
        metrics_list.append(metrics)
        print(f"    Run {run+1} — TSS={metrics['tss']:.4f}  F1={metrics['f1']:.4f}  "
              f"Train={metrics['train_time']:.2f}s")

    filepath = os.path.join(RESULTS_DIR, f"svm_{norm_key}.txt")
    save_results(metrics_list, filepath)
    print_results(metrics_list, f"SVM | {norm_label}")

print(f"\n\n✅  All SVM experiments complete.")
print(f"    Results saved to : {os.path.abspath(RESULTS_DIR)}/")
print(f"    Files : {sorted([f for f in os.listdir(RESULTS_DIR) if f.endswith('.txt')])}")

════════════════════════════════════════════════════════════
  Classifier : SVM
  Model      : LinearSVC  C=1  cw=balanced
════════════════════════════════════════════════════════════

  ── Dataset: hybrid_144 ──
    Run 1 — TSS=0.2434  F1=0.0556  Train=169.18s
    Run 2 — TSS=0.2434  F1=0.0556  Train=174.22s

───────────────────────────────────────────────────────
  SVM | hybrid_144
───────────────────────────────────────────────────────
  Run 1: TP=12  TN=3139  FP=386  FN=22
         TSS=0.2434  HSS1=0.0386  HSS2=0.0386  GSS=0.0197
         Recall=0.3529  F1=0.0556  Acc=0.8854
         Train=169.18s  Infer=0.0030s

  Run 2: TP=12  TN=3139  FP=386  FN=22
         TSS=0.2434  HSS1=0.0386  HSS2=0.0386  GSS=0.0197
         Recall=0.3529  F1=0.0556  Acc=0.8854
         Train=174.22s  Infer=0.0009s

  ── Average of 2 runs ──
  tss          : 0.2434
  hss1         : 0.0386
  hss2         : 0.0386
  gss          : 0.0197
  recall       : 0.3529
  f1           : 0.0556
  accuracy     : 0.8854

### GRU 

In [3]:
# ══════════════════════════════════════════════════════════════
# HELPER FUNCTIONS — PyTorch
# ══════════════════════════════════════════════════════════════
import torch
import torch.nn as nn
import numpy as np
import time
import os
import warnings
warnings.filterwarnings('ignore')

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")


class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1, dropout=0.3):
        super(GRUModel, self).__init__()
        self.gru      = nn.GRU(input_size, hidden_size, num_layers=num_layers,
                               batch_first=True,
                               dropout=dropout if num_layers > 1 else 0)
        self.bn       = nn.BatchNorm1d(hidden_size)
        self.dropout  = nn.Dropout(dropout)
        self.fc1      = nn.Linear(hidden_size, 32)
        self.relu     = nn.ReLU()
        self.dropout2 = nn.Dropout(0.2)
        self.fc2      = nn.Linear(32, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        out    = out[:, -1, :]
        out    = self.bn(out)
        out    = self.dropout(out)
        out    = self.relu(self.fc1(out))
        out    = self.dropout2(out)
        return self.fc2(out).squeeze()


def train_and_evaluate_gru(params, X_train, y_train, X_val, y_val, X_test, y_test, class_ratio=13):
    X_tr = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_tr = torch.tensor(y_train, dtype=torch.float32).to(device)
    X_va = torch.tensor(X_val,   dtype=torch.float32).to(device)
    y_va = torch.tensor(y_val,   dtype=torch.float32).to(device)
    X_te = torch.tensor(X_test,  dtype=torch.float32).to(device)

    input_size = X_train.shape[2]
    model      = GRUModel(input_size,
                          hidden_size = params["units"],
                          num_layers  = params["layers"],
                          dropout     = params["dropout"]).to(device)

    pos_weight = torch.tensor([float(class_ratio)], dtype=torch.float32).to(device)
    criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer  = torch.optim.Adam(model.parameters(), lr=params["lr"])

    batch_size     = params["batch_size"]
    n_samples      = X_tr.shape[0]
    best_val_loss  = float('inf')
    patience_count = 0
    best_state     = None

    t0 = time.time()
    for epoch in range(50):
        model.train()
        perm = torch.randperm(n_samples)
        for i in range(0, n_samples, batch_size):
            idx    = perm[i:i + batch_size]
            xb, yb = X_tr[idx], y_tr[idx]
            optimizer.zero_grad()
            loss   = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_va), y_va).item()

        if val_loss < best_val_loss:
            best_val_loss  = val_loss
            patience_count = 0
            best_state     = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience_count += 1
            if patience_count >= 5:
                break

    train_time = time.time() - t0

    model.load_state_dict(best_state)
    model.eval()
    t0 = time.time()
    with torch.no_grad():
        y_prob = torch.sigmoid(model(X_te)).cpu().numpy()
    y_pred     = (y_prob >= 0.5).astype(int)
    infer_time = time.time() - t0

    metrics = compute_metrics(y_test.astype(int), y_pred)
    metrics['train_time'] = train_time
    metrics['infer_time'] = infer_time
    return metrics, model


Using device: mps


In [4]:
# ══════════════════════════════════════════════════════════════
# LOAD TIME SERIES DATASETS (3D) — hybrid timestep variants
# ══════════════════════════════════════════════════════════════
import pickle
import numpy as np

BASE = "./final_split_data_HybridNorm_timesteps"
TS_DATA_PATHS = {
    "hybrid_144": {
        "train": f"{BASE}/hybrid_144/train_set.pkl",
        "val":   f"{BASE}/hybrid_144/val_set.pkl",
        "test":  f"{BASE}/hybrid_144/test_set.pkl",
    },
    "hybrid_72": {
        "train": f"{BASE}/hybrid_72/train_set.pkl",
        "val":   f"{BASE}/hybrid_72/val_set.pkl",
        "test":  f"{BASE}/hybrid_72/test_set.pkl",
    },
}

def load_ts_split(path):
    with open(path, "rb") as f:
        data = pickle.load(f)
    return data["X"].astype(np.float32), data["y"].astype(np.float32)

datasets_gru = {}
for norm_name, splits in TS_DATA_PATHS.items():
    X_train, y_train = load_ts_split(splits["train"])
    X_val,   y_val   = load_ts_split(splits["val"])
    X_test,  y_test  = load_ts_split(splits["test"])
    datasets_gru[norm_name] = {
        "X_train": X_train, "y_train": y_train,
        "X_val":   X_val,   "y_val":   y_val,
        "X_test":  X_test,  "y_test":  y_test,
    }
    print(f"  {norm_name:<15}  train={X_train.shape}  val={X_val.shape}  test={X_test.shape}")




  hybrid_144       train=(12455, 144, 10)  val=(1780, 144, 10)  test=(3559, 144, 10)
  hybrid_72        train=(12455, 72, 10)  val=(1780, 72, 10)  test=(3559, 72, 10)


In [6]:
# ══════════════════════════════════════════════════════════════
# RUN EXPERIMENTS
# ══════════════════════════════════════════════════════════════
best_params = {"units": 128, "dropout": 0.3, "layers": 1, "lr": 0.001, "batch_size": 32}
DATASET_NAMES = {
    "hybrid_144": "hybrid_144",
    "hybrid_72":  "hybrid_72",
}

print(f"{'═'*60}")
print(f"  Classifier : GRU (PyTorch)")
print(f"  Best params: {best_params}")
print(f"{'═'*60}")

for norm_key, norm_label in DATASET_NAMES.items():
    if norm_key not in datasets_gru:
        print(f"  [{norm_key}] not found — skipping")
        continue

    d  = datasets_gru[norm_key]
    cr = int((d["y_train"] == 0).sum() / (d["y_train"] == 1).sum())
    print(f"\n  ── Dataset: {norm_label}  (class ratio {cr}:1) ──")

    metrics_list = []
    for run in range(2):
        print(f"    Run {run+1} …", flush=True)
        metrics, _ = train_and_evaluate_gru(
            best_params,
            d["X_train"], d["y_train"],
            d["X_val"],   d["y_val"],
            d["X_test"],  d["y_test"],
            class_ratio = cr
        )
        metrics_list.append(metrics)
        print(f"    Run {run+1} — TSS={metrics['tss']:.4f}  F1={metrics['f1']:.4f}  "
              f"Train={metrics['train_time']:.1f}s")

    filepath = os.path.join(RESULTS_DIR, f"gru_{norm_key}.txt")
    save_results(metrics_list, filepath)
    print_results(metrics_list, f"GRU | {norm_label}")

print(f"\n\n✅  All GRU experiments complete.")
print(f"    Results saved to : {os.path.abspath(RESULTS_DIR)}/")
print(f"    Files : {sorted([f for f in os.listdir(RESULTS_DIR) if f.endswith('.txt')])}")

════════════════════════════════════════════════════════════
  Classifier : GRU (PyTorch)
  Best params: {'units': 128, 'dropout': 0.3, 'layers': 1, 'lr': 0.001, 'batch_size': 32}
════════════════════════════════════════════════════════════

  ── Dataset: hybrid_144  (class ratio 104:1) ──
    Run 1 …
    Run 1 — TSS=0.5023  F1=0.1120  Train=220.4s
    Run 2 …
    Run 2 — TSS=0.5192  F1=0.1045  Train=144.6s

───────────────────────────────────────────────────────
  GRU | hybrid_144
───────────────────────────────────────────────────────
  Run 1: TP=20  TN=3222  FP=303  FN=14
         TSS=0.5023  HSS1=0.0964  HSS2=0.0964  GSS=0.0507
         Recall=0.5882  F1=0.1120  Acc=0.9109
         Train=220.37s  Infer=0.1858s

  Run 2: TP=21  TN=3178  FP=347  FN=13
         TSS=0.5192  HSS1=0.0885  HSS2=0.0885  GSS=0.0463
         Recall=0.6176  F1=0.1045  Acc=0.8988
         Train=144.61s  Infer=0.1855s

  ── Average of 2 runs ──
  tss          : 0.5107
  hss1         : 0.0925
  hss2         : 0.